In [70]:
from pydantic import BaseModel

# 1. Define the parameters for a single generator
class Generator(BaseModel):
    name: str
    cost_per_mw: float
    max_capacity_mw: float
    min_capacity_mw: float = 0.0

# 2. Define the overall problem structure
class DispatchProblem(BaseModel):
    objective_type: str  # e.g., "minimize cost"
    total_demand_mw: float
    generators: list[Generator]

In [66]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load the hidden API key from your .env file
load_dotenv()

# 2. Initialize the Gemini client 
client = genai.Client()

# 3. Define the plain-text problem (Unstructured Data)
prompt_text = """
The grid operator is facing a peak demand period and needs to meet a total demand of 500 MW. There are four generators connected to the grid. 

Generator 1 is a large nuclear plant; it costs $15 per MW to run, has a maximum capacity of 300 MW, and because it cannot easily power down, it has a strict minimum capacity of 150 MW. 
Generator 2 is a coal plant that costs $25 per MW, with a maximum capacity of 250 MW and a minimum stable generation of 50 MW. 
Generator 3 is a highly flexible gas plant costing $40 per MW with a maximum capacity of 150 MW. 
Generator 4 is a wind farm that costs only $5 per MW to operate and has a maximum output of 80 MW. 

The objective is to minimize the total generation cost.
"""

# 4. Ask Gemini to read the text and fill out the Pydantic form
print("Sending text to Gemini...")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DispatchProblem,
        temperature=0.0, # We set this to 0 because we want exact facts, not creativity
    ),
)

# 5. Convert the JSON response back into our Pydantic object
extracted_data = DispatchProblem.model_validate_json(response.text)

print("\n--- Extraction Success! ---")
print(f"Objective: {extracted_data.objective_type}")
print(f"Demand: {extracted_data.total_demand_mw} MW")
for gen in extracted_data.generators:
    print(f"Found {gen.name}: Min {gen.min_capacity_mw}MW, Max {gen.max_capacity_mw}MW, Cost ${gen.cost_per_mw}/MW")



Sending text to Gemini...

--- Extraction Success! ---
Objective: minimize_cost
Demand: 500.0 MW
Found Nuclear Plant: Min 150.0MW, Max 300.0MW, Cost $15.0/MW
Found Coal Plant: Min 50.0MW, Max 250.0MW, Cost $25.0/MW
Found Gas Plant: Min 0.0MW, Max 150.0MW, Cost $40.0/MW
Found Wind Farm: Min 0.0MW, Max 80.0MW, Cost $5.0/MW


In [69]:
import pyomo.environ as pyo
model = pyo.ConcreteModel()
gen_names = [g.name for g in extracted_data.generators]
print(gen_names)
model.generator = pyo.Set(initialize = gen_names)
model.total_demand_mw = pyo.Param(initialize = extracted_data.total_demand_mw)
model.gen_power = pyo.Var(model.generator)

costs = {g.name: g.cost_per_mw for g in extracted_data.generators}
model.cost = pyo.Param(model.generator, initialize=costs)
min_capacity = {g.name: g.min_capacity_mw for g in extracted_data.generators}
model.min_capicity = pyo.Param(model.generator, initialize=min_capacity)
max_capacity = {g.name: g.max_capacity_mw for g in extracted_data.generators}
model.max_capicity = pyo.Param(model.generator, initialize=max_capacity)

def max_rule(model, g):
    return model.gen_power[g] <= model.max_capicity[g]
def min_rule(model, g):
    return model.gen_power[g] >= model.min_capicity[g]
model.c1 = pyo.Constraint(expr = sum(model.gen_power[g] for g in model.generator)== model.total_demand_mw)
model.c2 = pyo.Constraint(model.generator, rule = max_rule)
model.c3 = pyo.Constraint(model.generator, rule = min_rule)

model.f1 = pyo.Objective(expr = sum(model.cost[g] * model.gen_power[g] for g in model.generator), sense=pyo.minimize)

solver = pyo.SolverFactory('glpk')
results = solver.solve(model)
for g in model.generator:
    print(f"{g}: {pyo.value(model.gen_power[g])} MW")



['Nuclear Plant', 'Coal Plant', 'Gas Plant', 'Wind Farm']
Nuclear Plant: 300.0 MW
Coal Plant: 120.0 MW
Gas Plant: 0.0 MW
Wind Farm: 80.0 MW
